# Multi-Agent Patterns

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) LangGraph roadmap.

You will learn five multi-agent patterns — Subagents, Handoffs, Skills, Router, and Custom Workflow — and when to use each.

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain "langchain[openai]"

## 2. Set Up Your API Key

Enter your OpenAI API key when prompted. The key is not stored or displayed.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Subagents Pattern

Parent agent invokes child agents as tools.

In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.prebuilt import ToolNode
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

llm = ChatOpenAI(model="gpt-4o-mini")

@tool
def research_agent(query: str) -> str:
    """Research a topic and return findings."""
    response = llm.invoke([
        ("system", "You are a research assistant. Provide concise findings."),
        ("human", query),
    ])
    return response.content

@tool
def math_agent(expression: str) -> str:
    """Evaluate a math expression."""
    response = llm.invoke([
        ("system", "You are a math expert. Solve the expression step by step."),
        ("human", expression),
    ])
    return response.content

tools = [research_agent, math_agent]
llm_with_tools = llm.bind_tools(tools)

class SubagentState(TypedDict):
    messages: Annotated[list, add_messages]

def parent_agent(state: SubagentState) -> dict:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

def should_continue(state: SubagentState) -> str:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return "end"

graph = StateGraph(SubagentState)
graph.add_node("agent", parent_agent)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
graph.add_edge("tools", "agent")

app = graph.compile()

result = app.invoke({"messages": [("human", "What is 42 * 17?")]})
print(result["messages"][-1].content)

## 4. Handoff Pattern

Agents transfer control to each other using `Command(goto=...)`.

In [ ]:
from langgraph.types import Command

class HandoffState(TypedDict):
    messages: Annotated[list, add_messages]
    current_agent: str

def sales_agent(state: HandoffState) -> Command:
    response = llm.invoke([
        ("system",
         "You are a sales agent. Answer pricing questions. "
         "If the user asks a technical question, say 'transferring to support'."),
        *state["messages"],
    ])
    if "transferring to support" in response.content.lower():
        return Command(
            update={"messages": [response], "current_agent": "support"},
            goto="support",
        )
    return Command(
        update={"messages": [response], "current_agent": "sales"},
        goto=END,
    )

def support_agent(state: HandoffState) -> Command:
    response = llm.invoke([
        ("system",
         "You are a support agent. Answer technical questions. "
         "If the user asks about pricing, say 'transferring to sales'."),
        *state["messages"],
    ])
    if "transferring to sales" in response.content.lower():
        return Command(
            update={"messages": [response], "current_agent": "sales"},
            goto="sales",
        )
    return Command(
        update={"messages": [response], "current_agent": "support"},
        goto=END,
    )

def router(state: HandoffState) -> Command:
    response = llm.invoke([
        ("system", "Route to 'sales' for pricing/purchase or 'support' for technical help. Reply with one word."),
        *state["messages"],
    ])
    dest = "sales" if "sales" in response.content.lower() else "support"
    return Command(
        update={"current_agent": dest},
        goto=dest,
    )

graph2 = StateGraph(HandoffState)
graph2.add_node("router", router)
graph2.add_node("sales", sales_agent)
graph2.add_node("support", support_agent)
graph2.add_edge(START, "router")

app2 = graph2.compile()

result = app2.invoke({"messages": [("human", "How much does the enterprise plan cost?")]})
print(f"Routed to: {result['current_agent']}")
print(f"Response: {result['messages'][-1].content[:200]}...")

## 5. Router with Parallel Execution

A central router dispatches independent subtasks to specialists in parallel via `Send`.

In [ ]:
import operator
from langgraph.types import Send

class RouterState(TypedDict):
    query: str
    results: Annotated[list[str], operator.add]

class AgentInput(TypedDict):
    query: str
    agent_type: str

def router_node(state: RouterState) -> dict:
    return {}

def dispatch(state: RouterState) -> list[Send]:
    return [
        Send("specialist", {"query": state["query"], "agent_type": "researcher"}),
        Send("specialist", {"query": state["query"], "agent_type": "analyst"}),
        Send("specialist", {"query": state["query"], "agent_type": "writer"}),
    ]

def specialist(state: AgentInput) -> dict:
    response = llm.invoke([
        ("system", f"You are a {state['agent_type']}. Respond concisely."),
        ("human", state["query"]),
    ])
    return {"results": [f"[{state['agent_type']}]: {response.content[:100]}"]}

graph3 = StateGraph(RouterState)
graph3.add_node("router", router_node)
graph3.add_node("specialist", specialist)
graph3.add_conditional_edges("router", dispatch, path_map=["specialist"])
graph3.add_edge(START, "router")
graph3.add_edge("specialist", END)

app3 = graph3.compile()

result = app3.invoke({"query": "What is retrieval-augmented generation?"})
print(f"Specialists dispatched: {len(result['results'])}")
for r in result["results"]:
    print(f"\n{r}")

## 6. Custom Workflow Pattern

Hardcoded graph topology with agents at each node for predictable, auditable pipelines.

In [ ]:
class PipelineState(TypedDict):
    messages: Annotated[list, add_messages]
    draft: str
    review: str
    final: str

def drafter(state: PipelineState) -> dict:
    response = llm.invoke([
        ("system", "Write a first draft based on the request."),
        *state["messages"],
    ])
    return {"draft": response.content}

def reviewer(state: PipelineState) -> dict:
    response = llm.invoke([
        ("system", "Review this draft and suggest improvements."),
        ("human", state["draft"]),
    ])
    return {"review": response.content}

def editor(state: PipelineState) -> dict:
    response = llm.invoke([
        ("system", "Produce the final version incorporating the review."),
        ("human", f"Draft:\n{state['draft']}\n\nReview:\n{state['review']}"),
    ])
    return {"final": response.content}

graph4 = StateGraph(PipelineState)
graph4.add_node("drafter", drafter)
graph4.add_node("reviewer", reviewer)
graph4.add_node("editor", editor)
graph4.add_edge(START, "drafter")
graph4.add_edge("drafter", "reviewer")
graph4.add_edge("reviewer", "editor")
graph4.add_edge("editor", END)

app4 = graph4.compile()

result = app4.invoke({"messages": [("human", "Write a product description for an AI coding assistant")]})
print(f"Draft length: {len(result['draft'])} chars")
print(f"Review length: {len(result['review'])} chars")
print(f"Final length: {len(result['final'])} chars")
print(f"\nFinal:\n{result['final'][:300]}...")

## What You Learned

1. **Subagents** invoke child agents as tools — parent controls all flow
2. **Handoffs** let agents transfer control via `Command(goto=...)` — agents decide routing
3. **Router** dispatches to specialists in parallel via `Send` — for independent subtasks
4. **Custom Workflow** hardcodes the topology — predictable and auditable pipelines
5. Choose the pattern based on autonomy needs, task independence, and predictability requirements